# PyTorch Fundamentals: Zero to Confident
**A Professional Deep Dive from First Principles to Fashion-MNIST Production Classifier**

Welcome to PyTorch Fundamentals! This notebook is your comprehensive, step-by-step masterclass in PyTorch. Whether you are running on an ordinary CPU or an 8GB NVIDIA GPU (like an RTX 4060), this notebook automatically detects your hardware and guides you through core deep learning engineering practices.

---

## Pedagogical Glossary
Here is our master reference table of terms. Every technical word is defined in plain language when it is first encountered.

| Term | One-Sentence Plain Definition |
| :--- | :--- |
| **Tensor** | A multi-dimensional grid of numbers (like a list, a spreadsheet, or a 3D cube) stored in computer memory. |
| **Rank / Dimension (`ndim`)** | The number of directions or axes along which numbers are arranged in a tensor. |
| **Shape** | A tuple of integers stating exactly how many numbers exist along each axis of a tensor. |
| **Data Type (`dtype`)** | The binary format and precision used to store each number in memory (e.g., 32-bit float, 64-bit integer). |
| **Indexing** | Selecting specific individual numbers or slices from a tensor using numeric coordinates. |
| **Slicing** | Extracting a sub-window of elements along an axis using start, stop, and step bounds (`start:stop:step`). |
| **Broadcasting** | PyTorch's rule-based stretching of smaller tensors across dimensions of size 1 so arithmetic works without copying data. |
| **Stride** | The number of memory steps needed to jump to the next element along each dimension in physical RAM. |
| **View** | A new tensor metadata window looking at the exact same physical memory buffer with a different shape or stride. |
| **Reshape** | A method that returns a view if memory is contiguous, or creates a memory copy if it is not. |
| **Contiguity** | Storing tensor elements sequentially in memory in the exact order traversed by row-major indexing. |
| **Device** | The physical hardware processor and memory where a tensor resides (either system RAM `cpu` or graphics VRAM `cuda:0`). |
| **NumPy Interoperability** | The ability to share memory directly between PyTorch CPU tensors and NumPy ndarrays without overhead. |


## Pedagogical Glossary
Here is our master reference table of terms. Every technical word is defined in plain language when it is first encountered.

| Term | One-Sentence Plain Definition |
| :--- | :--- |
| **Tensor** | A multi-dimensional grid of numbers (like a list, a spreadsheet, or a 3D cube) stored in computer memory. |
| **Rank / Dimension (`ndim`)** | The number of directions or axes along which numbers are arranged in a tensor. |
| **Shape** | A tuple of integers stating exactly how many numbers exist along each axis of a tensor. |
| **Data Type (`dtype`)** | The binary format and precision used to store each number in memory (e.g., 32-bit float, 64-bit integer). |
| **Indexing** | Selecting specific individual numbers or slices from a tensor using numeric coordinates. |
| **Slicing** | Extracting a sub-window of elements along an axis using start, stop, and step bounds (`start:stop:step`). |
| **Broadcasting** | PyTorch's rule-based stretching of smaller tensors across dimensions of size 1 so arithmetic works without copying data. |
| **Stride** | The number of memory steps needed to jump to the next element along each dimension in physical RAM. |
| **View** | A new tensor metadata window looking at the exact same physical memory buffer with a different shape or stride. |
| **Reshape** | A method that returns a view if memory is contiguous, or creates a memory copy if it is not. |
| **Contiguity** | Storing tensor elements sequentially in memory in the exact order traversed by row-major indexing. |
| **Device** | The physical hardware processor and memory where a tensor resides (either system RAM `cpu` or graphics VRAM `cuda:0`). |
| **NumPy Interoperability** | The ability to share memory directly between PyTorch CPU tensors and NumPy ndarrays without overhead. |
| **Autograd** | PyTorch's automatic differentiation engine that records math operations and calculates exact derivatives. |
| **Computational Graph** | A directed flowchart of nodes (tensors) and edges (math operations) that tracks how outputs were produced. |
| **Dynamic Graph (Define-by-Run)** | A computational graph built step-by-step at runtime as code executes, allowing dynamic Python loops and conditionals. |
| **`requires_grad`** | A tensor flag that tells PyTorch to record every math operation applied to it for derivative calculations. |
| **Backward Pass (`.backward()`)** | Traversing the computational graph in reverse to calculate the gradient of a final scalar with respect to all leaf tensors. |
| **Gradient (`.grad`)** | A tensor storing the rate of change of the loss with respect to a parameter, indicating how to tweak it to reduce error. |
| **Leaf Tensor** | A root tensor created directly by the user (like weights or inputs), rather than produced by an intermediate math operation. |
| **`torch.no_grad()`** | A context manager that disables graph building to save memory and speed up evaluation during inference. |
| **`.detach()`** | A method that disconnects a tensor from the computational graph so gradients can never flow backwards through it. |
| **`nn.Module`** | The base class for all neural network blueprints in PyTorch, managing layer hierarchy and learnable parameters. |
| **`__init__` vs `forward`** | `__init__` instantiates and allocates layer parameters; `forward` defines the calculation path when data flows through. |
| **Linear Layer (`nn.Linear`)** | A fully connected layer that applies the affine transformation $y = x W^T + b$ to input features. |
| **Activation Function (ReLU)** | A non-linear element-wise function $\max(0, x)$ that enables neural networks to learn non-linear patterns. |
| **Dropout (`nn.Dropout`)** | A regularization technique that randomly zeroes out activations during training to prevent co-adaptation of neurons. |
| **Logits** | The raw, unnormalized scalar scores output by the final linear layer of a classification model before softmax. |
| **`parameters()`** | An iterator yielding all learnable weight and bias tensors registered inside a module. |
| **`state_dict()`** | A standard Python dictionary mapping the string name of every parameter and buffer to its underlying tensor. |


In [1]:
# 1. Import core PyTorch library
import torch
# 2. Import numerical computing library for array conversions
import numpy as np
# 3. Import plotting library for visual diagnostic plots
import matplotlib.pyplot as plt

# 4. Check if CUDA (GPU support) is available on this system
use_cuda = torch.cuda.is_available()
# 5. Automatically select the GPU device if available, otherwise fallback to CPU
device = torch.device("cuda:0" if use_cuda else "cpu")

# 6. Print system configuration details
print(f"PyTorch Version: {torch.__version__}")
print(f"Device Selected: {device}")
if use_cuda:
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


PyTorch Version: 2.14.0+cu130
Device Selected: cuda:0
GPU Hardware: NVIDIA GeForce RTX 4060 Laptop GPU
Available VRAM: 8.00 GB


**What we just saw:**
PyTorch confirmed our execution device. If you have an NVIDIA GPU, `device` will point to `cuda:0` with its VRAM capacity reported; otherwise, it cleanly defaults to your system `cpu`.

---
# Step 1: Tensors — Creation, Shapes, Dtypes, Indexing, Broadcasting, Views, GPU & NumPy

Tensors are the fundamental data structure in PyTorch. Everything from input image pixels to layer weights and loss gradients is represented as a tensor.


### 1.1 Tensor Creation
**1. The idea in ONE plain sentence:**
A tensor is a multi-dimensional array of numbers stored in contiguous blocks of computer memory.

**2. Real-life analogy:**
Think of a tensor as an ice-cube tray where each individual pocket holds a single number. A 1D tray is a single row, a 2D tray is a flat rectangular tray, and a 3D tray is a stack of rectangular trays inside a freezer.

**3. Tiny example worked by hand:**
- A scalar (0D): just the number `7.0`.
- A 1D vector of length 3: `[1.0, 2.0, 3.0]`.
- A 2D matrix of shape (2, 3):
  $$\begin{bmatrix} 1.0 & 2.0 & 3.0 \\ 4.0 & 5.0 & 6.0 \end{bmatrix}$$
  It has 2 rows and 3 columns, making 6 numbers in total.

**4. The math with every symbol explained in words:**
A tensor $T \in \mathbb{R}^{d_1 \times d_2 \times \dots \times d_k}$ is a mathematical object indexed by $k$ coordinates $(i_1, i_2, \dots, i_k)$.
- $\mathbb{R}$: Real numbers (numbers with decimal points).
- $k$: The rank (number of dimensions or axes).
- $d_m$: The size (length) along axis $m$.
- Total elements $N = \prod_{m=1}^k d_m$ (the product of all dimension lengths).

**5. Code explained line by line:**
Let's create tensors from Python lists, and using built-in generators (`zeros`, `ones`, `rand`).


In [2]:
# 1. Create a 1D tensor directly from a Python list
t_1d = torch.tensor([1.0, 2.0, 3.0])
# 2. Create a 2D tensor (matrix) from nested Python lists
t_2d = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
# 3. Create a tensor filled with zeros of shape (2, 3)
t_zeros = torch.zeros(2, 3)
# 4. Create a tensor filled with ones of shape (2, 3)
t_ones = torch.ones(2, 3)
# 5. Create a tensor with uniformly distributed random numbers in [0, 1)
t_rand = torch.rand(2, 3)

# 6. Display tensor values and their exact shapes
print("t_1d shape:", t_1d.shape, "| elements:", t_1d)
print("t_2d shape:", t_2d.shape, "| elements:\n", t_2d)
print("t_zeros shape:", t_zeros.shape)
print("t_rand shape:", t_rand.shape)


t_1d shape: torch.Size([3]) | elements: tensor([1., 2., 3.])
t_2d shape: torch.Size([2, 3]) | elements:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
t_zeros shape: torch.Size([2, 3])
t_rand shape: torch.Size([2, 3])


**What we just saw:**
We created 1D and 2D tensors. Notice that `torch.tensor` takes data as an input list, while helper functions like `torch.zeros` take dimension sizes.

**6. One common beginner mistake:**
Passing numbers as separate arguments instead of a list into `torch.tensor`. Writing `torch.tensor(1, 2, 3)` causes a `TypeError` because PyTorch expects a single sequence or array, like `torch.tensor([1, 2, 3])`.

**7. Quick "check yourself" question:**
*Question:* What is the total number of elements in a tensor of shape `(3, 4, 5)`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> $3 \times 4 \times 5 = 60$ elements.
</details>

**Short Exercise 1.1:**
Create a 3D tensor of shape `(2, 3, 4)` initialized with all ones, and compute its total element count using `.numel()`.


In [3]:
# Exercise 1.1: Create 3D tensor and verify number of elements
# 1. Initialize a (2, 3, 4) tensor with ones
ex_ones = torch.ones(2, 3, 4)
# 2. Count total elements using the .numel() method
total_elements = ex_ones.numel()
# 3. Print the shape and total elements
print("Exercise 1.1 shape:", ex_ones.shape)
print("Exercise 1.1 total elements (numel):", total_elements)
assert total_elements == 24, "Expected 24 elements!"


Exercise 1.1 shape: torch.Size([2, 3, 4])
Exercise 1.1 total elements (numel): 24


### 1.2 Shapes and Dimensions: The NCHW Layout in Computer Vision
**1. The idea in ONE plain sentence:**
Shape tells you how many numbers are stacked along each direction, which in deep vision follows the standard order: Batch size, Channels, Height, and Width ($B \times C \times H \times W$).

**2. Real-life analogy:**
Imagine a shipping crate holding photo albums:
- $B$ (Batch): The number of separate photo albums in the crate.
- $C$ (Channels): The number of color layers per page (e.g., 1 for grayscale, 3 for Red/Green/Blue).
- $H$ (Height): The number of rows of pixels from top to bottom.
- $W$ (Width): The number of columns of pixels from left to right.

**3. Tiny example worked by hand:**
For Fashion-MNIST, images are grayscale ($C = 1$) and $28 \times 28$ pixels ($H = 28, W = 28$).
If we load a mini-batch of 64 images, our tensor shape is:
$$(B, C, H, W) = (64, 1, 28, 28)$$
Total pixel values in this batch = $64 \times 1 \times 28 \times 28 = 50,176$.

**4. The math with every symbol explained in words:**
$$X \in \mathbb{R}^{B \times C \times H \times W}$$
- $B$: Batch dimension index $b \in \{0, \dots, B-1\}$ (which sample).
- $C$: Channel dimension index $c \in \{0, \dots, C-1\}$ (which color band).
- $H$: Height coordinate $h \in \{0, \dots, H-1\}$ (vertical row index).
- $W$: Width coordinate $w \in \{0, \dots, W-1\}$ (horizontal column index).

**5. Code explained line by line:**


In [4]:
# 1. Simulate a batch of 8 Fashion-MNIST images (B=8, C=1, H=28, W=28)
batch_images = torch.rand(8, 1, 28, 28)

# 2. Inspect shape tuple
print("Batch shape:", batch_images.shape)
# 3. Inspect individual dimensions
b, c, h, w = batch_images.shape
print(f"Batch size (B): {b} images")
print(f"Channels (C): {c} channel (grayscale)")
print(f"Height (H): {h} pixel rows")
print(f"Width (W): {w} pixel columns")
# 4. Check rank / number of dimensions
print("Number of dimensions (ndim):", batch_images.ndim)


Batch shape: torch.Size([8, 1, 28, 28])
Batch size (B): 8 images
Channels (C): 1 channel (grayscale)
Height (H): 28 pixel rows
Width (W): 28 pixel columns
Number of dimensions (ndim): 4


**What we just saw:**
We simulated a 4D batch tensor and unpacked its 4 dimensions. In PyTorch vision pipelines, the channel dimension always precedes height and width ($B, C, H, W$), unlike standard NumPy/OpenCV image arrays which often store $(H, W, C)$.

**6. One common beginner mistake:**
Mixing up Height and Width or confusing $(B, C, H, W)$ with $(B, H, W, C)$. Passing an image batch shaped $(64, 28, 28, 1)$ into `torch.nn.Conv2d` will fail with an error complaining that input channels do not match!

**7. Quick "check yourself" question:**
*Question:* If you have 16 RGB images that are 64 pixels high and 32 pixels wide, what is the exact shape of this tensor in PyTorch?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Shape is <code>(16, 3, 64, 32)</code>: $B=16$, $C=3$ (RGB), $H=64$, $W=32$.
</details>

**Short Exercise 1.2:**
Create a dummy batch representing 4 color (RGB) images of size $32 \times 32$. Print its shape and verify that the channel dimension is 3.


In [5]:
# Exercise 1.2: Create a dummy RGB image batch
# 1. Create tensor of shape (B=4, C=3, H=32, W=32)
rgb_batch = torch.zeros(4, 3, 32, 32)
# 2. Verify shape and channel count
print("RGB batch shape:", rgb_batch.shape)
assert rgb_batch.shape[1] == 3, "Channel dimension must be 3 for RGB!"


RGB batch shape: torch.Size([4, 3, 32, 32])


### 1.3 Data Types (dtypes)
**1. The idea in ONE plain sentence:**
Data type (`dtype`) dictates the binary representation and memory precision used to store each individual number.

**2. Real-life analogy:**
Think of choosing between lined index cards: an integer card only accepts whole numbers (like counting items: 0, 1, 2), while a floating-point card has decimal precision allowing fine fractions (like 0.8371).

**3. Tiny example worked by hand:**
- `torch.float32`: 32 bits (4 bytes) per number. Standard default for neural network weights and activations.
- `torch.int64` (or `torch.long`): 64 bits (8 bytes) per whole number. Required for classification class indices in `nn.CrossEntropyLoss`.
- `torch.float16`: 16 bits (2 bytes). Used in mixed-precision GPU acceleration to halve VRAM usage.

**4. The math with every symbol explained in words:**
Memory footprint:
$$\text{Memory (bytes)} = N \times S$$
- $N$: Total number of elements.
- $S$: Byte size per element ($4$ bytes for float32, $8$ bytes for int64, $2$ bytes for float16).
For 1,000,000 float32 numbers: $1{,}000{,}000 \times 4 = 4{,}000{,}000\text{ bytes} \approx 3.81\text{ MB}$.

**5. Code explained line by line:**


In [6]:
# 1. Create a floating point tensor (default is float32)
f_tensor = torch.tensor([1.5, 2.7, 3.2])
# 2. Inspect default dtype
print("Float tensor dtype:", f_tensor.dtype)

# 3. Create an integer tensor (default from Python ints is int64 / long)
i_tensor = torch.tensor([0, 1, 9])
print("Integer tensor dtype:", i_tensor.dtype)

# 4. Explicitly convert float tensor to 16-bit half precision
half_tensor = f_tensor.to(torch.float16)
print("Half precision dtype:", half_tensor.dtype)

# 5. Explicitly convert integer labels to long (standard for cross entropy targets)
target_tensor = i_tensor.long()
print("Target labels dtype:", target_tensor.dtype)


Float tensor dtype: torch.float32
Integer tensor dtype: torch.int64
Half precision dtype: torch.float16
Target labels dtype: torch.int64


**What we just saw:**
We checked default types and used `.to(...)` and `.long()` to convert between datatypes.

**6. One common beginner mistake:**
Passing float labels (e.g. `torch.tensor([0.0, 1.0])`) to `nn.CrossEntropyLoss`. PyTorch will crash with `RuntimeError: Expected floating point type for input with exception of long for target`. Classification targets must always be `torch.int64` (`torch.long`).

**7. Quick "check yourself" question:**
*Question:* How much memory in megabytes does a tensor of $10^7$ numbers consume when stored in `float32` vs `float16`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> In float32 (4 bytes/element), $10^7 \times 4 = 40\text{ MB}$. In float16 (2 bytes/element), $10^7 \times 2 = 20\text{ MB}$ (50% memory savings!).
</details>

**Short Exercise 1.3:**
Create an integer tensor `[0, 2, 4]` and cast it to `torch.float32`. Print its dtype before and after.


In [7]:
# Exercise 1.3: Convert integer tensor to float32
# 1. Create integer tensor
ex_int = torch.tensor([0, 2, 4])
print("Before cast:", ex_int.dtype)
# 2. Cast to float32 using .float()
ex_float = ex_int.float()
print("After cast:", ex_float.dtype)
assert ex_float.dtype == torch.float32, "Expected torch.float32!"


Before cast: torch.int64
After cast: torch.float32


### 1.4 Indexing and Slicing
**1. The idea in ONE plain sentence:**
Indexing and slicing allow you to access single elements or contiguous sub-regions of a tensor along any dimension.

**2. Real-life analogy:**
Think of street addresses: city (dimension 0), street (dimension 1), house number (dimension 2). Specifying `[2, :, 0]` means: pick city 2, visit every street (`:`), and knock on house number 0.

**3. Tiny example worked by hand:**
Given a $3 \times 3$ grid:
$$\begin{bmatrix} 10 & 20 & 30 \\ 40 & 50 & 60 \\ 70 & 80 & 90 \end{bmatrix}$$
- Element at row 1, col 2: `[1, 2]` $\rightarrow 60$.
- First two rows, all columns: `[0:2, :]` $\rightarrow \begin{bmatrix} 10 & 20 & 30 \\ 40 & 50 & 60 \end{bmatrix}$.
- Column 0 across all rows: `[:, 0]` $\rightarrow [10, 40, 70]$.

**4. The math with every symbol explained in words:**
Sub-slice $T[s_0:e_0, s_1:e_1]$ selects indices:
- $i_0 \in [s_0, e_0)$ along axis 0.
- $i_1 \in [s_1, e_1)$ along axis 1.
Notice that the end index is always non-inclusive (half-open interval $[s, e)$).

**5. Code explained line by line:**


In [8]:
# 1. Create a 3x3 matrix with concrete values
mat = torch.tensor([[10.0, 20.0, 30.0],
                    [40.0, 50.0, 60.0],
                    [70.0, 80.0, 90.0]])

# 2. Extract single item at row 1, column 2
item = mat[1, 2]
print("Single item at mat[1, 2]:", item.item())

# 3. Extract the entire first column (all rows, column 0)
col_0 = mat[:, 0]
print("First column shape:", col_0.shape, "| values:", col_0)

# 4. Extract top 2x2 submatrix
sub_mat = mat[0:2, 0:2]
print("Top 2x2 submatrix shape:", sub_mat.shape)
print(sub_mat)


Single item at mat[1, 2]: 60.0
First column shape: torch.Size([3]) | values: tensor([10., 40., 70.])
Top 2x2 submatrix shape: torch.Size([2, 2])
tensor([[10., 20.],
        [40., 50.]])


**What we just saw:**
We indexed individual elements and sliced submatrices. Note `.item()` extracts a 0D tensor as a standard Python float.

**6. One common beginner mistake:**
Forgetting that basic slicing creates a *view* that shares underlying memory with the original tensor. If you modify `sub_mat[0, 0] = 999.0`, the original `mat[0, 0]` also turns into 999.0! If you need an isolated copy, call `.clone()`.

**7. Quick "check yourself" question:**
*Question:* Given a batch of images `x` with shape `(64, 1, 28, 28)`, what slice expression extracts only the very first image while keeping all 4 dimensions intact?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> <code>x[0:1, :, :, :]</code> (or <code>x[0:1]</code>), which returns shape <code>(1, 1, 28, 28)</code>. (Using <code>x[0]</code> drops the batch dimension, returning <code>(1, 28, 28)</code>).
</details>

**Short Exercise 1.4:**
Extract the last row of `mat` using negative indexing (`mat[-1, :]`) and verify its shape is `(3,)`.


In [9]:
# Exercise 1.4: Extract the last row using negative indexing
# 1. Slice last row
last_row = mat[-1, :]
# 2. Print values and shape
print("Last row values:", last_row)
print("Last row shape:", last_row.shape)
assert last_row.shape == torch.Size([3]), "Shape must be (3,)!"


Last row values: tensor([70., 80., 90.])
Last row shape: torch.Size([3])


### 1.5 Broadcasting
**1. The idea in ONE plain sentence:**
Broadcasting automatically duplicates dimensions of length 1 during arithmetic so tensors of different shapes can be combined without copying memory.

**2. Real-life analogy:**
Imagine a teacher applying a curved grade adjustment: the teacher writes `+5 points` once on the blackboard (a scalar or single row), and it automatically gets added to every single student's exam score on the roster.

**3. Tiny example worked by hand:**
Add a row matrix $A$ of shape $(1, 3)$ and a column matrix $B$ of shape $(2, 1)$:
$$A = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}, \quad B = \begin{bmatrix} 10 \\ 20 \end{bmatrix}$$
Broadcasting expands $A$ along rows and $B$ along columns:
$$A_{\text{expanded}} = \begin{bmatrix} 1 & 2 & 3 \\ 1 & 2 & 3 \end{bmatrix}, \quad B_{\text{expanded}} = \begin{bmatrix} 10 & 10 & 10 \\ 20 & 20 & 20 \end{bmatrix}$$
$$A + B = \begin{bmatrix} 1+10 & 2+10 & 3+10 \\ 1+20 & 2+20 & 3+20 \end{bmatrix} = \begin{bmatrix} 11 & 12 & 13 \\ 21 & 22 & 23 \end{bmatrix} \quad \text{shape } (2, 3)$$

**4. The math with every symbol explained in words:**
Two shapes $(a_1, a_2, \dots)$ and $(b_1, b_2, \dots)$ are compatible for broadcasting if, comparing dimensions from right to left:
- They are equal ($a_i = b_i$), OR
- One of them is $1$ ($a_i = 1$ or $b_i = 1$), OR
- One dimension does not exist (it is implicitly treated as $1$).

**5. Code explained line by line:**


In [10]:
# 1. Create row tensor of shape (1, 3)
row_vec = torch.tensor([[1.0, 2.0, 3.0]])
# 2. Create column tensor of shape (2, 1)
col_vec = torch.tensor([[10.0], [20.0]])

# 3. Add them together with automatic broadcasting
broadcasted_sum = row_vec + col_vec

# 4. Print input shapes and output shape/result
print("row_vec shape:", row_vec.shape)
print("col_vec shape:", col_vec.shape)
print("broadcasted_sum shape:", broadcasted_sum.shape)
print("Resulting matrix:")
print(broadcasted_sum)


row_vec shape: torch.Size([1, 3])
col_vec shape: torch.Size([2, 1])
broadcasted_sum shape: torch.Size([2, 3])
Resulting matrix:
tensor([[11., 12., 13.],
        [21., 22., 23.]])


**What we just saw:**
PyTorch automatically aligned the shapes $(1, 3)$ and $(2, 1)$ to produce $(2, 3)$ matching our hand calculation.

**6. One common beginner mistake:**
Accidentally broadcasting a 1D vector of predictions of shape `(N,)` with targets of shape `(N, 1)`. When you compute `loss = pred - target`, PyTorch does NOT throw an error; instead, it broadcasts them into a giant $(N, N)$ matrix! If $N=10{,}000$, it creates a $100{,}000{,}000$-element matrix, quietly exhausting memory. Always ensure shapes match before computing loss!

**7. Quick "check yourself" question:**
*Question:* Can a tensor of shape `(5, 1, 4)` and a tensor of shape `(1, 3, 4)` be added via broadcasting? What is the output shape?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Yes! Comparing dimensions right to left: $4=4$ (match), $1$ and $3$ (compatible $\rightarrow 3$), $5$ and $1$ (compatible $\rightarrow 5$). Output shape is <code>(5, 3, 4)</code>.
</details>

**Short Exercise 1.5:**
Given a batch of image features `features = torch.ones(4, 10)` and a bias vector `bias = torch.tensor([0.5] * 10)`, add them and verify the output shape is `(4, 10)`.


In [11]:
# Exercise 1.5: Broadcast bias vector across feature batch
# 1. Batch of 4 samples with 10 features each
features = torch.ones(4, 10)
# 2. Bias vector with 10 features (shape (10,))
bias = torch.full((10,), 0.5)
# 3. Add bias to features
features_with_bias = features + bias
# 4. Confirm shape and values
print("features_with_bias shape:", features_with_bias.shape)
print("Sample row:", features_with_bias[0])
assert features_with_bias.shape == torch.Size([4, 10]), "Shape must remain (4, 10)!"


features_with_bias shape: torch.Size([4, 10])
Sample row: tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000])


### 1.6 Reshape vs View (and Memory Contiguity)
**1. The idea in ONE plain sentence:**
Changing the shape of a tensor can either create a zero-cost "view" looking at existing memory or create a new memory copy if the elements are not sequentially laid out in RAM.

**2. Real-life analogy:**
Imagine 12 toy blocks sitting in a straight row on a table. Calling `.view(3, 4)` is like simply saying "let's count every 4 blocks as a new group." You didn't move a single block. But if someone scrambles the blocks across the room (transposing), you must physically pick them up and re-align them in order before grouping them again.

**3. Tiny example worked by hand:**
Take a $2 \times 3$ matrix:
$$\begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}$$
In physical memory, the numbers sit in linear row-major order: `[1, 2, 3, 4, 5, 6]`.
- Flattening to shape `(6,)` via `.view(6)` just changes the shape label.
- Transposing to shape `(3, 2)` gives:
  $$\begin{bmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{bmatrix}$$
  The elements in memory are still physically stored as `[1, 2, 3, 4, 5, 6]`, but the stride changed. Calling `.view()` directly on this transposed tensor fails because memory is non-contiguous!

**4. The math with every symbol explained in words:**
- **Stride:** For a tensor $T$, `stride[i]` is the number of memory addresses skipped to move by $1$ along dimension $i$.
- A tensor is **contiguous** if:
  $$\text{stride}[k] = 1 \quad \text{and} \quad \text{stride}[i] = \text{stride}[i+1] \times \text{size}[i+1]$$
- `.reshape()` checks: if contiguous, it calls `.view()`; if non-contiguous, it calls `.contiguous().view()` automatically.

**5. Code explained line by line:**


In [12]:
# 1. Create a contiguous 2x3 tensor
t_orig = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("t_orig is_contiguous:", t_orig.is_contiguous())
print("t_orig strides:", t_orig.stride())

# 2. Reshape to (6,) using .view() (zero memory copy)
t_flat_view = t_orig.view(6)
print("t_flat_view shape:", t_flat_view.shape)

# 3. Transpose matrix (swaps rows and columns)
t_transposed = t_orig.t()
print("t_transposed shape:", t_transposed.shape)
print("t_transposed is_contiguous:", t_transposed.is_contiguous())

# 4. Demonstrate why .reshape() is safer than .view()
# Calling .view(6) on t_transposed would throw RuntimeError!
t_safe_flat = t_transposed.reshape(6)
print("t_safe_flat succeeds! Shape:", t_safe_flat.shape, "| Values:", t_safe_flat)


t_orig is_contiguous: True
t_orig strides: (3, 1)
t_flat_view shape: torch.Size([6])
t_transposed shape: torch.Size([3, 2])
t_transposed is_contiguous: False
t_safe_flat succeeds! Shape: torch.Size([6]) | Values: tensor([1, 4, 2, 5, 3, 6])


**What we just saw:**
Transposing changed the strides without moving memory, rendering `t_transposed` non-contiguous (`is_contiguous() == False`). Calling `.reshape()` handled this cleanly by re-arranging memory under the hood.

**6. One common beginner mistake:**
Calling `.view(...)` after transpose or permutation operations without `.contiguous()`. It crashes with: `RuntimeError: view size is not compatible with input tensor's size and stride (tensor is not contiguous)`. Using `.reshape(...)` or `.contiguous().view(...)` fixes this error every time.

**7. Quick "check yourself" question:**
*Question:* When should you strictly prefer `.view()` over `.reshape()`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Use <code>.view()</code> when you want an absolute architectural guarantee that no memory copy is created (for strict zero-copy efficiency). If a copy is ever required, <code>.view()</code> intentionally errors out so you know immediately. Otherwise, <code>.reshape()</code> is the safer and more flexible choice.
</details>

**Short Exercise 1.6:**
Flatten a batch of dummy images of shape `(16, 1, 28, 28)` into a 2D matrix of shape `(16, 784)` using `.reshape()`.


In [13]:
# Exercise 1.6: Flatten image batch for linear layers
# 1. Create dummy batch (B=16, C=1, H=28, W=28)
dummy_batch = torch.rand(16, 1, 28, 28)
# 2. Flatten spatial dimensions into 784 features per sample
flattened_batch = dummy_batch.reshape(16, -1)
# 3. Inspect shape (-1 automatically computed 1*28*28 = 784)
print("Flattened batch shape:", flattened_batch.shape)
assert flattened_batch.shape == torch.Size([16, 784]), "Shape must be (16, 784)!"


Flattened batch shape: torch.Size([16, 784])


### 1.7 Moving Between CPU and GPU (Automatic Device Selection)
**1. The idea in ONE plain sentence:**
Tensors live in either system RAM (`cpu`) or graphics card memory (`cuda`), and operations only work between tensors living on the exact same device.

**2. Real-life analogy:**
Think of a master chef in the kitchen (GPU) and a bookkeeper in the front office (CPU). You cannot mix flour and eggs if the flour is on the kitchen counter and the eggs are locked inside the bookkeeper's desk. You must bring all ingredients into the kitchen first.

**3. Tiny example worked by hand:**
- CPU tensor $A$ lives in system RAM.
- GPU tensor $B$ lives in dedicated GDDR VRAM.
- $A + B$ $\rightarrow$ PyTorch raises a device mismatch error.
- Move $A$ to GPU: $A_{\text{gpu}} = A\text{.to}(\text{"cuda"})$.
- Now $A_{\text{gpu}} + B$ executes in parallel across thousands of GPU cores.

**4. The math / Hardware architecture:**
Data transfer occurs across the PCIe bus:
$$\text{Transfer Time} = \frac{\text{Tensor Bytes}}{\text{PCIe Bandwidth}}$$
While PCIe transfer has a small overhead, once tensors reside in GPU VRAM, matrix operations run 10x-100x faster than on CPU.

**5. Code explained line by line:**


In [14]:
# 1. Create a tensor on CPU
cpu_tensor = torch.tensor([1.0, 2.0, 3.0])
print("Initial device:", cpu_tensor.device)

# 2. Move tensor to our globally selected device (cuda or cpu)
dev_tensor = cpu_tensor.to(device)
print("Device after .to(device):", dev_tensor.device)

# 3. Move back to CPU explicitly
back_to_cpu = dev_tensor.cpu()
print("Device after .cpu():", back_to_cpu.device)

# 4. Check whether tensor is currently on CUDA
print("Is tensor on CUDA?:", dev_tensor.is_cuda)


Initial device: cpu
Device after .to(device): cuda:0
Device after .cpu(): cpu
Is tensor on CUDA?: True


**What we just saw:**
We moved tensors between CPU and GPU with `.to(device)`. The `.is_cuda` attribute confirms whether a tensor lives in graphics VRAM.

**6. One common beginner mistake:**
Calling `model.to(device)` but forgetting to call `images = images.to(device)` in the training loop. This produces the infamous error: `RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!`.

**7. Quick "check yourself" question:**
*Question:* Does `tensor.to(device)` modify `tensor` in place, or does it return a new tensor?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> For tensors, <code>.to(device)</code> returns a <b>new copy</b> on the target device! You must write <code>tensor = tensor.to(device)</code>. (Note: for <code>nn.Module</code> models, <code>model.to(device)</code> modifies the model in place, which makes this a very frequent beginner trap!).
</details>

**Short Exercise 1.7:**
Write a small helper function `ensure_device(tensor, target_device)` that checks if a tensor is already on `target_device`, and moves it only if necessary.


In [15]:
# Exercise 1.7: Device transfer helper function
def ensure_device(tensor: torch.Tensor, target_device: torch.device) -> torch.Tensor:
    # 1. Check if tensor already resides on target_device
    if tensor.device != target_device:
        # 2. Move tensor to target device
        return tensor.to(target_device)
    # 3. Return original tensor without copying
    return tensor

# Test helper function
sample_t = torch.tensor([42.0])
verified_t = ensure_device(sample_t, device)
print(f"Verified device: {verified_t.device}")
assert verified_t.device == device, "Device mismatch!"


Verified device: cuda:0


### 1.8 NumPy Interoperability
**1. The idea in ONE plain sentence:**
PyTorch CPU tensors and NumPy arrays can convert back and forth without copying memory because they share the exact same underlying RAM buffer.

**2. Real-life analogy:**
Putting two different name tags on the same person: one tag says "PyTorch Tensor" and the other says "NumPy Array". If that person changes their shirt, both name tags show the change because there is only one physical person.

**3. Tiny example worked by hand:**
- Create NumPy array: `a = np.array([1.0, 2.0])`.
- Convert to PyTorch: `t = torch.from_numpy(a)`.
- Change `a[0] = 99.0`.
- Look at `t[0]`: it is automatically `99.0` because no memory was duplicated!

**4. The math / Memory pointer:**
Both the PyTorch tensor descriptor and the NumPy array descriptor store pointers pointing to the exact same physical memory address in host RAM:
$$\text{address}(t.\text{data\_ptr}()) == \text{address}(a.__\text{array\_interface__}['\text{data}'][0])$$

**5. Code explained line by line:**


In [16]:
# 1. Create a NumPy array
np_arr = np.array([10.0, 20.0, 30.0], dtype=np.float32)

# 2. Convert to PyTorch tensor (shares memory!)
t_from_np = torch.from_numpy(np_arr)
print("Initial PyTorch tensor from NumPy:", t_from_np)

# 3. Modify NumPy array in place
np_arr[0] = 999.0

# 4. Observe that the PyTorch tensor changed simultaneously!
print("PyTorch tensor after NumPy modification:", t_from_np)

# 5. Convert PyTorch tensor back to NumPy
np_back = t_from_np.numpy()
print("Converted back to NumPy type:", type(np_back), "| values:", np_back)


Initial PyTorch tensor from NumPy: tensor([10., 20., 30.])
PyTorch tensor after NumPy modification: tensor([999.,  20.,  30.])
Converted back to NumPy type: <class 'numpy.ndarray'> | values: [999.  20.  30.]


**What we just saw:**
Modifying the NumPy array immediately updated the PyTorch tensor because they share memory. Calling `.numpy()` converts back to NumPy instantly.

**6. One common beginner mistake:**
Calling `.numpy()` directly on a GPU tensor. PyTorch raises: `TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() first`. NumPy cannot access GPU VRAM directly; you must always call `tensor.cpu().numpy()`.

**7. Quick "check yourself" question:**
*Question:* How do you convert a PyTorch tensor to NumPy WITHOUT sharing memory so future edits don't affect each other?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Call <code>tensor.clone().numpy()</code> (or <code>np_arr.copy()</code>). The <code>.clone()</code> method allocates a brand new memory buffer.
</details>

**Short Exercise 1.8:**
Convert a GPU tensor to a NumPy array safely using `.detach().cpu().numpy()`.


In [17]:
# Exercise 1.8: Safe GPU-to-NumPy conversion
# 1. Create a tensor on our device
gpu_sample = torch.tensor([1.23, 4.56]).to(device)
# 2. Safely move to CPU and convert to NumPy
np_result = gpu_sample.detach().cpu().numpy()
# 3. Inspect result
print("NumPy result from device tensor:", np_result, "| Type:", type(np_result))
assert isinstance(np_result, np.ndarray), "Expected NumPy ndarray!"


NumPy result from device tensor: [1.23 4.56] | Type: <class 'numpy.ndarray'>


### Step 1 Recap
- **Recap in 3 bullets:**
  1. Tensors are multi-dimensional grids of numbers characterized by their shape (e.g. $B, C, H, W$ in vision) and data type (`float32` for weights, `long` for labels).
  2. Broadcasting expands length-1 dimensions automatically, while `.view()`/`.reshape()` re-indexes memory layout; use `.reshape()` to avoid contiguity pitfalls.
  3. Tensors must reside on the same device (`cpu` or `cuda`) to interact; PyTorch CPU tensors share memory directly with NumPy arrays.

- **If you remember only one thing:**
  *Always verify tensor shapes and devices at every boundary: mismatched dimensions or mismatched devices cause 90% of beginner PyTorch bugs.*


---
# Step 2: Autograd & Dynamic Computational Graphs
How PyTorch tracks mathematical operations, builds graphs on the fly, and computes exact derivatives automatically.


### 2.0 Prerequisite Refresher: Derivatives & The Chain Rule
Before looking at code, here is a 5-line refresher on the core math behind deep learning:
1. **Derivative (Rate of Change):** If $y = f(x)$, the derivative $\frac{dy}{dx}$ tells you how much $y$ changes when you nudge $x$ by a tiny amount.
2. **Power Rule:** The derivative of $x^n$ is $n x^{n-1}$. For example, if $f(x) = x^2$, then $\frac{df}{dx} = 2x$.
3. **Constant Multiple Rule:** The derivative of $c \cdot f(x)$ is $c \cdot \frac{df}{dx}$. For $2x^2$, the derivative is $2(2x) = 4x$.
4. **Sum Rule:** The derivative of $f(x) + g(x)$ is simply $\frac{df}{dx} + \frac{dg}{dx}$.
5. **Chain Rule:** If $z$ depends on $y$ and $y$ depends on $x$, then $\frac{dz}{dx} = \frac{dz}{dy} \times \frac{dy}{dx}$. Multiplying local nudge rates gives the overall rate of change!


### 2.1 Dynamic Computational Graphs (Define-by-Run)
**1. The idea in ONE plain sentence:**
A computational graph is a flowchart of math operations that PyTorch constructs dynamically as your Python code executes, allowing regular Python `if` statements and loops to alter the network architecture on the fly.

**2. Real-life analogy:**
Think of leaving breadcrumbs as you explore a hiking trail. You don't have to map out every single fork in the forest beforehand; you just walk wherever you like, and your breadcrumbs record your exact path so you can retrace your steps backwards at sunset.

**3. Tiny example worked by hand:**
Take two numbers $a = 2.0$ and $b = 3.0$:
- Operation 1: $c = a \times b = 2.0 \times 3.0 = 6.0$.
- Operation 2: $d = c + 4.0 = 6.0 + 4.0 = 10.0$.
PyTorch creates two nodes ($c$ with function `MulBackward0`, and $d$ with function `AddBackward0`).

**4. The math with every symbol explained in words:**
A computational graph is a Directed Acyclic Graph (DAG) $G = (V, E)$:
- Vertices $V$: Tensors ($a, b, c, d$).
- Edges $E$: Mathematical operations linking inputs to outputs.
- In PyTorch, every intermediate tensor has a `.grad_fn` attribute holding a reference to the backward function that generated it.

**5. Code explained line by line:**


In [18]:
# 1. Create two leaf tensors with gradient tracking enabled
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

# 2. Perform multiplication operation
c = a * b
# 3. Perform addition operation
d = c + 4.0

# 4. Inspect .grad_fn on intermediate and final nodes
print("Leaf tensor 'a' grad_fn:", a.grad_fn)
print("Intermediate 'c' grad_fn:", c.grad_fn)
print("Final output 'd' grad_fn:", d.grad_fn)
print("Value of d:", d.item())


Leaf tensor 'a' grad_fn: None
Intermediate 'c' grad_fn: <MulBackward0 object at 0x0000022C32C69BA0>
Final output 'd' grad_fn: <AddBackward0 object at 0x0000022C5F3461A0>
Value of d: 10.0


**What we just saw:**
Leaf inputs created directly by us have `grad_fn=None`. Intermediate tensors produced by math have specific gradient functions (`MulBackward0`, `AddBackward0`) attached to them, forming the computational graph.

**6. One common beginner mistake:**
Thinking the graph remains alive in memory forever. Once you call `.backward()`, PyTorch immediately destroys the intermediate graph buffers to free RAM! If you try calling `.backward()` a second time on the same output, PyTorch throws: `RuntimeError: Trying to backward through the graph a second time`.

**7. Quick "check yourself" question:**
*Question:* What is the difference between a "static graph" (like older TensorFlow 1.x) and PyTorch's "dynamic graph"?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> A static graph must be compiled ahead of time like a blueprint before any data flows through it. PyTorch's dynamic graph (define-by-run) is constructed on the fly during execution, so standard Python control flow (like <code>for</code> loops and <code>if</code> statements) works natively.
</details>

**Short Exercise 2.1:**
Write a dynamic condition: if an input $x > 0$, compute $y = x^2$, otherwise compute $y = x^3$. Verify that PyTorch attaches the correct `grad_fn`.


In [19]:
# Exercise 2.1: Dynamic control flow with Autograd
# 1. Define leaf tensor with positive value
x_dyn = torch.tensor(2.0, requires_grad=True)
# 2. Dynamic branch based on value
if x_dyn > 0:
    y_dyn = x_dyn ** 2
else:
    y_dyn = x_dyn ** 3

# 3. Inspect grad_fn
print("Dynamic y value:", y_dyn.item())
print("Dynamic grad_fn:", y_dyn.grad_fn)
assert "PowBackward" in str(y_dyn.grad_fn), "Expected power backward function!"


Dynamic y value: 4.0
Dynamic grad_fn: <PowBackward0 object at 0x0000022C5F346140>


### 2.2 Hand Gradient Calculation vs Autograd Matching Proof
**1. The idea in ONE plain sentence:**
Autograd applies the chain rule backward through every node in the graph, producing exact analytical derivatives that match manual calculus.

**2. Real-life analogy:**
Think of a pocket calculator where, after giving you the answer to an equation, it can tell you exactly how much your result would change if you tweaked any single input by 1%.

**3. Tiny example worked by hand with real numbers:**
Let us choose a quadratic polynomial:
$$y = 2x^2 + 5x + 1$$
We pick a concrete input value:
$$x = 3.0$$

**Step A: Forward Pass (Calculate output value)**
1. $x = 3.0$
2. $x^2 = 3.0^2 = 9.0$
3. $2x^2 = 2 \times 9.0 = 18.0$
4. $5x = 5 \times 3.0 = 15.0$
5. $y = 18.0 + 15.0 + 1.0 = 34.0$

**Step B: Analytical Derivative by Hand (Calculus)**
1. Derivative of $2x^2$ with respect to $x$ is $2 \times (2x) = 4x$.
2. Derivative of $5x$ with respect to $x$ is $5$.
3. Derivative of constant $1$ is $0$.
4. Total derivative:
   $$\frac{dy}{dx} = 4x + 5$$
5. Evaluate at $x = 3.0$:
   $$\left.\frac{dy}{dx}\right|_{x=3.0} = 4(3.0) + 5 = 12.0 + 5.0 = 17.0$$

**Step C: PyTorch Autograd Verification**
Now let us write this exact polynomial in PyTorch, call `.backward()`, and verify `x.grad == 17.0`!

**4. The math with every symbol explained in words:**
$$\frac{dy}{dx} = \lim_{\Delta x \to 0} \frac{y(x + \Delta x) - y(x)}{\Delta x} = 4x + 5$$
- $\frac{dy}{dx}$: Rate of change of $y$ with respect to $x$.
- $x$: The independent input variable.
- $x.grad$: PyTorch's storage attribute holding the accumulated value of $\frac{dy}{dx}$.

**5. Code explained line by line:**


In [20]:
# 1. Initialize input x = 3.0 with requires_grad=True
x = torch.tensor(3.0, requires_grad=True)

# 2. Compute forward pass: y = 2*x^2 + 5*x + 1
y = 2 * (x ** 2) + 5 * x + 1.0

# 3. Print computed forward value (should be 34.0)
print(f"Forward pass output y = {y.item()} (Hand calculation: 34.0)")

# 4. Execute backward pass to compute dy/dx
y.backward()

# 5. Extract calculated gradient from x.grad
computed_grad = x.grad.item()
print(f"Autograd x.grad       = {computed_grad} (Hand calculation: 17.0)")

# 6. Formally verify exact match
assert computed_grad == 17.0, f"Mismatch! Expected 17.0, got {computed_grad}"
print("VERIFICATION SUCCESSFUL: Autograd exactly matches hand calculus!")


Forward pass output y = 34.0 (Hand calculation: 34.0)
Autograd x.grad       = 17.0 (Hand calculation: 17.0)
VERIFICATION SUCCESSFUL: Autograd exactly matches hand calculus!


**What we just saw:**
PyTorch's `.backward()` computed $\frac{dy}{dx} = 17.0$ matching our pencil-and-paper calculus calculation down to the last decimal place!

**6. One common beginner mistake:**
Calling `.backward()` on a non-scalar tensor without passing an argument. If $y$ has shape `(2, 2)`, calling `y.backward()` fails with `RuntimeError: grad can be implicitly created only for scalar outputs`. In neural networks, loss is always reduced to a single scalar (e.g. mean loss) before `.backward()` is called!

**7. Quick "check yourself" question:**
*Question:* What would `x.grad` be before calling `y.backward()`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> <code>None</code>! PyTorch does not allocate memory for <code>.grad</code> until <code>.backward()</code> is explicitly executed.
</details>

**Short Exercise 2.2:**
Let $w = 4.0$ with `requires_grad=True`, and $L = 3w^3 - 2w$. Compute $\frac{dL}{dw}$ by hand at $w=4.0$, run Autograd, and verify the match.


In [21]:
# Exercise 2.2: Hand calculation vs Autograd for L = 3*w^3 - 2*w
# By hand: dL/dw = 9*w^2 - 2. At w=4: 9*(16) - 2 = 144 - 2 = 142.0
# 1. Initialize w = 4.0
w = torch.tensor(4.0, requires_grad=True)
# 2. Forward pass
L = 3 * (w ** 3) - 2 * w
# 3. Backward pass
L.backward()
# 4. Verify match
print(f"w.grad = {w.grad.item()} | Hand calculation = 142.0")
assert w.grad.item() == 142.0, "Expected 142.0!"


w.grad = 142.0 | Hand calculation = 142.0


### 2.3 Disabling Autograd: `torch.no_grad()`
**1. The idea in ONE plain sentence:**
The `torch.no_grad()` context manager temporarily deactivates Autograd's tracking engine, reducing memory usage and speeding up calculations during model evaluation.

**2. Real-life analogy:**
Think of shopping with a notebook where you usually write down the price and location of every item so you can return them later. When you are just browsing with no intention of returning anything, you leave the notebook in your pocket—saving time and paper.

**3. Tiny example worked by hand:**
- With tracking: $x$ ($requires\_grad=True$) $\times 2 \rightarrow$ result tensor has $requires\_grad=True$ and a $MulBackward$ graph node.
- Inside `with torch.no_grad()`: $x \times 2 \rightarrow$ result tensor has $requires\_grad=False$ and $grad\_fn=None$. Zero graph memory allocated.

**4. The math / Computational cost:**
During training, forward pass activations must be cached in memory to compute backward gradients:
$$\text{Memory}(\text{Training}) = \text{Model Weights} + \text{Optimizer State} + \text{Intermediate Activations}$$
During evaluation with `torch.no_grad()`, intermediate activations can be discarded immediately:
$$\text{Memory}(\text{Inference}) = \text{Model Weights}$$

**5. Code explained line by line:**


In [22]:
# 1. Leaf parameter with gradient tracking
param = torch.tensor([5.0], requires_grad=True)

# 2. Standard operation outside context manager
out_normal = param * 2.0
print("Normal operation requires_grad:", out_normal.requires_grad)
print("Normal operation grad_fn:", out_normal.grad_fn)

# 3. Same operation inside torch.no_grad() context manager
with torch.no_grad():
    out_no_grad = param * 2.0
    print("Inside no_grad requires_grad:", out_no_grad.requires_grad)
    print("Inside no_grad grad_fn:", out_no_grad.grad_fn)


Normal operation requires_grad: True
Normal operation grad_fn: <MulBackward0 object at 0x0000022C5F203A00>
Inside no_grad requires_grad: False
Inside no_grad grad_fn: None


**What we just saw:**
Inside `torch.no_grad()`, `out_no_grad.requires_grad` was forced to `False` and `grad_fn` was `None`, ensuring zero graph tracking overhead.

**6. One common beginner mistake:**
Evaluating a model or running a validation loop without wrapping it in `with torch.no_grad():`. The forward pass keeps piling up intermediate activation graphs for hundreds of validation batches until the GPU runs out of VRAM and crashes!

**7. Quick "check yourself" question:**
*Question:* Does `with torch.no_grad():` modify the underlying `param.requires_grad` attribute of your model weights?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> No! The model weights still have <code>requires_grad=True</code>; only the <i>new outputs produced inside the block</i> are prevented from recording graph history.
</details>

**Short Exercise 2.3:**
Verify that doing math inside `torch.no_grad()` does not allow calling `.backward()`.


In [23]:
# Exercise 2.3: Verify backward failure inside no_grad
p = torch.tensor([10.0], requires_grad=True)
with torch.no_grad():
    res = p * 3.0

try:
    res.backward()
    print("Backward unexpectedly succeeded!")
except RuntimeError as e:
    print("Expected RuntimeError caught successfully:")
    print("Error message:", str(e).split('\n')[0])


Expected RuntimeError caught successfully:
Error message: element 0 of tensors does not require grad and does not have a grad_fn


### 2.4 Detaching Tensors: `.detach()`
**1. The idea in ONE plain sentence:**
The `.detach()` method creates a new tensor sharing the same data that is severed from the computational graph, preventing gradients from flowing back past this point.

**2. Real-life analogy:**
Cutting a kite's string. The kite still hovers with the exact same shape and position, but pulling on your handle will no longer exert any pull on the kite.

**3. Tiny example worked by hand:**
Let $u = 3.0$ with $requires\_grad=True$ and $w = 2.0$ with $requires\_grad=True$.
Let $v = u^2 = 9.0$.
Compute $z = v.\text{detach}() \times w = 9.0 \times 2.0 = 18.0$.
When we backpropagate from $z$:
- Rate of change with respect to $w$ is $v = 9.0$ ($w.\text{grad} = 9.0$).
- But because $v$ was detached, the gradient path back to $u$ is severed. So $u.\text{grad}$ remains `None`!

**4. The math with every symbol explained in words:**
Given a composite function $z = f(g(u))$:
$$\frac{\partial z}{\partial u} = \frac{\partial z}{\partial g} \cdot \frac{\partial g}{\partial u}$$
If we detach $g(u)$:
$$\frac{\partial g_{\text{detached}}}{\partial u} = 0 \implies \frac{\partial z}{\partial u} = 0$$

**5. Code explained line by line:**


In [24]:
# 1. Define two leaf parameters
u = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)

# 2. Compute intermediate v = u^2 (equals 9.0)
v = u ** 2

# 3. Compute z = v.detach() * w
# Because v is detached, gradients flow to w, but are cut off from u!
z = v.detach() * w
z.backward()

# 4. Check gradients: w gets gradient 9.0, while u gets None!
print("w.grad (received gradient):", w.grad.item())
print("u.grad (cut off by .detach()):", u.grad)
assert w.grad.item() == 9.0, "Expected w.grad to be 9.0!"
assert u.grad is None, "u should receive no gradient because graph was severed!"


w.grad (received gradient): 9.0
u.grad (cut off by .detach()): None


**What we just saw:**
Because $v$ was detached before computing $z$, backpropagation through $z$ could not reach $u$, leaving `u.grad` as `None`.

**6. One common beginner mistake:**
Calling `.detach()` on a tensor inside a layer where you actually wanted to train upstream weights, which accidentally freezes earlier layers of your network. Use `.detach()` when you want to convert a tensor to NumPy, plot it with matplotlib, or explicitly stop gradient flow (such as in target networks in reinforcement learning).

**7. Quick "check yourself" question:**
*Question:* If you modify `v_detached[0] = 99.0` in place, does the original tensor `v[0]` change?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Yes! <code>.detach()</code> shares underlying data memory with the original tensor; it only severs the Autograd history. If you need independent memory, use <code>tensor.detach().clone()</code>.
</details>

**Short Exercise 2.4:**
Create a leaf tensor `m = torch.tensor([2.0], requires_grad=True)`. Compute `n = m * 4`. Convert `n` to a NumPy array safely using `.detach().numpy()`.


In [25]:
# Exercise 2.4: Convert tracked tensor to NumPy using .detach()
# 1. Create tracked tensor
m = torch.tensor([2.0], requires_grad=True)
# 2. Compute tracked intermediate
n = m * 4.0
# 3. Safely detach and convert to NumPy
np_arr_from_graph = n.detach().numpy()
# 4. Print result
print("NumPy array extracted from graph:", np_arr_from_graph)
assert isinstance(np_arr_from_graph, np.ndarray), "Expected NumPy ndarray!"


NumPy array extracted from graph: [8.]


### Step 2 Recap
- **Recap in 3 bullets:**
  1. PyTorch builds a dynamic computational graph on the fly during the forward pass; intermediate tensors remember how they were made via `.grad_fn`.
  2. Calling `.backward()` traverses the graph backwards using the chain rule to accumulate exact analytical gradients into each leaf tensor's `.grad` attribute.
  3. `torch.no_grad()` prevents graph construction entirely for fast inference, while `.detach()` severs a specific tensor from the graph while sharing memory.

- **If you remember only one thing:**
  *Autograd is simply a calculator that records a diary of every step so it can retrace the chain rule backwards—disable the diary with `torch.no_grad()` whenever you are not training!*


---
# Step 3: `nn.Module` — Building Neural Networks the Professional Way
How PyTorch encapsulates layers, state, and computation into clean, modular classes.


### 3.1 Understanding `nn.Module`: `__init__` vs `forward`
**1. The idea in ONE plain sentence:**
`nn.Module` is PyTorch's master template class for neural networks, where `__init__` builds the architectural components and `forward` specifies how data travels through them.

**2. Real-life analogy:**
Think of building a factory:
- `__init__`: You buy the industrial machines (conveyor belts, sorting arms, packaging presses) and bolt them to the factory floor.
- `forward`: You turn the power switch on and define the assembly line path: raw materials go to Machine 1, then Machine 2, and exit as finished goods.

**3. Tiny example worked by hand:**
Our Fashion-MNIST classifier receives a flattened grayscale image of shape $(B, 784)$ and classifies it into one of 10 clothing categories:
- **Input:** $x \in \mathbb{R}^{B \times 784}$ ($28 \times 28 = 784$ pixels).
- **Layer 1 (Linear):** Maps $784 \to 128$ features.
  - Weights $W_1 \in \mathbb{R}^{128 \times 784}$, Bias $b_1 \in \mathbb{R}^{128}$.
  - Parameters: $128 \times 784 + 128 = 100{,}352 + 128 = 100{,}480$.
- **Activation (ReLU):** $\max(0, z_1)$ (0 parameters).
- **Layer 2 (Linear):** Maps $128 \to 64$ features.
  - Weights $W_2 \in \mathbb{R}^{64 \times 128}$, Bias $b_2 \in \mathbb{R}^{64}$.
  - Parameters: $64 \times 128 + 64 = 8{,}192 + 64 = 8{,}256$.
- **Activation (ReLU):** $\max(0, z_2)$ (0 parameters).
- **Regularization (Dropout):** Drops 20% of activations randomly during training (0 parameters).
- **Output Layer (Linear):** Maps $64 \to 10$ class logits.
  - Weights $W_3 \in \mathbb{R}^{10 \times 64}$, Bias $b_3 \in \mathbb{R}^{10}$.
  - Parameters: $10 \times 64 + 10 = 640 + 10 = 650$.
- **Total Network Parameters by Hand:**
  $$100{,}480 + 8{,}256 + 650 = 109{,}386 \text{ learnable parameters}$$

**4. The math with every symbol explained in words:**
For each fully connected linear layer:
$$y = x W^T + b$$
- $x$: Input feature matrix of shape $(B, D_{\text{in}})$.
- $W$: Weight matrix of shape $(D_{\text{out}}, D_{\text{in}})$. Transpose $W^T$ has shape $(D_{\text{in}}, D_{\text{out}})$.
- $b$: Bias vector of shape $(D_{\text{out}})$, broadcasted across all $B$ samples.
- $y$: Output activation matrix of shape $(B, D_{\text{out}})$.

**5. Code explained line by line:**


In [26]:
# 1. Import torch.nn which holds all neural network building blocks
import torch.nn as nn

# 2. Define our custom classifier inheriting from nn.Module
class FashionClassifier(nn.Module):
    def __init__(self, in_features: int = 784, num_classes: int = 10):
        # 3. Always call super().__init__() to register PyTorch internal hooks
        super().__init__()
        # 4. First linear layer: 784 inputs -> 128 hidden neurons
        self.fc1 = nn.Linear(in_features, 128)
        # 5. Non-linear ReLU activation function
        self.relu = nn.ReLU()
        # 6. Second linear layer: 128 -> 64 hidden neurons
        self.fc2 = nn.Linear(128, 64)
        # 7. Dropout layer with 20% drop probability for regularization
        self.dropout = nn.Dropout(p=0.2)
        # 8. Output classification head: 64 hidden neurons -> 10 class logits
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 9. Flatten spatial dimensions if 4D (B, C, H, W) -> 2D (B, 784)
        x = x.reshape(x.shape[0], -1)
        # 10. Pass through first linear layer and ReLU activation
        x = self.relu(self.fc1(x))
        # 11. Pass through second linear layer and ReLU activation
        x = self.relu(self.fc2(x))
        # 12. Apply dropout regularization
        x = self.dropout(x)
        # 13. Output raw class logits (no softmax here; CrossEntropyLoss does it!)
        logits = self.fc3(x)
        return logits


**What we just saw:**
We defined `FashionClassifier`. Notice that `__init__` instantiates the layers once, and `forward` specifies how tensors pass through them.

**6. One common beginner mistake:**
Calling `model.forward(x)` directly instead of `model(x)`. In PyTorch, calling `model(x)` invokes Python's `__call__` method, which executes forward pre-hooks, PyTorch profiling hooks, Autograd tracking, and forward post-hooks before running your code. Calling `model.forward(x)` bypasses all hooks silently!

**7. Quick "check yourself" question:**
*Question:* What happens if you forget to write `super().__init__()` in the first line of `__init__`?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> PyTorch will crash with <code>AttributeError: cannot assign module before Module.__init__() call</code> because <code>super().__init__()</code> initializes PyTorch's internal parameter and submodule registration dictionaries.
</details>

**Short Exercise 3.1:**
Instantiate `FashionClassifier` and run a dummy batch of 4 Fashion-MNIST images through it using `model(x)`. Print the output shape.


In [27]:
# Exercise 3.1: Instantiate model and test forward call
# 1. Instantiate the model
model = FashionClassifier()
# 2. Create dummy input batch of 4 images (B=4, C=1, H=28, W=28)
dummy_x = torch.rand(4, 1, 28, 28)
# 3. Call the model using __call__ syntax: model(dummy_x)
logits = model(dummy_x)
# 4. Verify output shape is (4, 10)
print("Output logits shape:", logits.shape)
assert logits.shape == torch.Size([4, 10]), "Output shape must be (4, 10)!"


Output logits shape: torch.Size([4, 10])


### 3.2 Inspecting `parameters()`, `named_parameters()`, and `state_dict()`
**1. The idea in ONE plain sentence:**
`parameters()` gives the optimizer access to all learnable weight tensors, while `state_dict()` provides a serializable Python dictionary of the model's entire learned memory.

**2. Real-life analogy:**
Think of a music band:
- `parameters()`: The list of active musicians who need to receive practice instructions from the conductor (the optimizer).
- `state_dict()`: The sheet music notebook recording the exact notes each musician plays so the band can recreate the exact song on another stage.

**3. Tiny example worked by hand:**
Earlier, we calculated by hand that `FashionClassifier` has $109{,}386$ parameters:
- `fc1.weight`: $(128, 784) = 100{,}352$
- `fc1.bias`: $(128,) = 128$
- `fc2.weight`: $(64, 128) = 8{,}192$
- `fc2.bias`: $(64,) = 64$
- `fc3.weight`: $(10, 64) = 640$
- `fc3.bias`: $(10,) = 10$
Sum $= 109{,}386$.
Let us verify this programmatically!

**4. The math with every symbol explained in words:**
Total trainable parameter count:
$$P = \sum_{p \in \Theta} \text{numel}(p)$$
- $\Theta$: The set of all parameters where $p.\text{requires\_grad} = \text{True}$.
- $\text{numel}(p)$: Product of dimension sizes of tensor $p$.

**5. Code explained line by line:**


In [28]:
# 1. Iterate through named parameters and inspect shapes
print("--- MODEL ARCHITECTURE PARAMETERS ---")
total_params = 0
for name, param in model.named_parameters():
    param_count = param.numel()
    total_params += param_count
    print(f"{name:15s} | Shape: {str(param.shape):16s} | Count: {param_count:7d}")

# 2. Verify match with hand calculation
print(f"Total Trainable Parameters: {total_params:,}")
assert total_params == 109386, f"Mismatch! Expected 109,386, got {total_params}"

# 3. Inspect state_dict keys
print("\n--- MODEL STATE DICT KEYS ---")
s_dict = model.state_dict()
for key in s_dict.keys():
    print(f"Key: {key:15s} | Tensor Shape: {s_dict[key].shape}")


--- MODEL ARCHITECTURE PARAMETERS ---
fc1.weight      | Shape: torch.Size([128, 784]) | Count:  100352
fc1.bias        | Shape: torch.Size([128]) | Count:     128
fc2.weight      | Shape: torch.Size([64, 128]) | Count:    8192
fc2.bias        | Shape: torch.Size([64]) | Count:      64
fc3.weight      | Shape: torch.Size([10, 64]) | Count:     640
fc3.bias        | Shape: torch.Size([10]) | Count:      10
Total Trainable Parameters: 109,386

--- MODEL STATE DICT KEYS ---
Key: fc1.weight      | Tensor Shape: torch.Size([128, 784])
Key: fc1.bias        | Tensor Shape: torch.Size([128])
Key: fc2.weight      | Tensor Shape: torch.Size([64, 128])
Key: fc2.bias        | Tensor Shape: torch.Size([64])
Key: fc3.weight      | Tensor Shape: torch.Size([10, 64])
Key: fc3.bias        | Tensor Shape: torch.Size([10])


**What we just saw:**
We verified that `model.named_parameters()` matches our manual hand calculation ($109{,}386$ parameters) and inspected the keys inside `model.state_dict()`.

**6. One common beginner mistake:**
Instantiating layers inside `forward` instead of `__init__`, like writing `def forward(self, x): return nn.Linear(784, 10)(x)`. When you do this, a brand new linear layer with freshly randomized weights is created on every single forward pass, so the model can NEVER learn anything! All layers with trainable state must be created inside `__init__`.

**7. Quick "check yourself" question:**
*Question:* Does `state_dict()` store Python code or just parameter numbers?
<details>
<summary><b>Click to reveal answer</b></summary>
<b>Answer:</b> Just numbers! A <code>state_dict</code> is a pure Python dictionary mapping string names to raw tensors. It contains zero Python code or class blueprints, which is why you must instantiate the class blueprint first before loading a <code>state_dict</code>.
</details>

**Short Exercise 3.2:**
Write a one-liner list comprehension to calculate total trainable parameters from `model.parameters()`.


In [29]:
# Exercise 3.2: One-liner parameter counter
# 1. Compute count using generator comprehension
counted_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# 2. Print result
print(f"Counted trainable parameters: {counted_params:,}")
assert counted_params == 109386, "Expected 109,386 parameters!"


Counted trainable parameters: 109,386


### Step 3 Recap
- **Recap in 3 bullets:**
  1. Subclass `nn.Module`, initialize layers in `__init__` after calling `super().__init__()`, and route tensors in `forward`.
  2. Always call the model as `model(x)` (which triggers PyTorch's `__call__` hooks), never call `model.forward(x)` directly.
  3. `parameters()` yields tensors for the optimizer to update, while `state_dict()` maps names to weight tensors for saving and loading.

- **If you remember only one thing:**
  *Define what layers exist in `__init__`, define how data flows through them in `forward`, and execute via `model(x)`!*
